# getitem-back-add-at — worked example 2: Backward of row gather (embedding backward)

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `getitem-back-add-at`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Gathering rows `out = x[idx]` from an `(N, D)` table has the same scatter-add backward, now adding whole rows of `grad_out` into the selected rows of a zeros tensor. This is precisely the embedding-layer backward: a token appearing multiple times accumulates its gradient.

## Worked solution

We compute the gradient for a row gather from an embedding table.

1. `x` is `(N, D)`, `idx` is `(K,)` row indices, `out` is `(K, D)`, and `grad_out` is `(K, D)`.
2. Each output row depends only on the corresponding source row, so `dL/dx[j, :] = sum over i with idx[i]==j of grad_out[i, :]` — a full-row contribution.
3. Allocate `grad_in = t.zeros_like(x)` of shape `(N, D)`.
4. `grad_in.index_add_(0, idx, grad_out)` scatter-adds each `grad_out` row into row `idx[i]` of `grad_in`. Repeated token ids accumulate, matching how an embedding gradient sums over every occurrence in the batch.
5. We use a repeated token id and verify its row equals the sum of the matching gradient rows.

In [ ]:
import torch as t

t.manual_seed(1)
x = t.zeros(4, 3)
idx = t.tensor([1, 3, 1])
grad_out = t.tensor([[1.0, 1.0, 1.0], [2.0, 2.0, 2.0], [4.0, 4.0, 4.0]])

def getitem_back_rows(grad_out, x, idx):
    grad_in = t.zeros_like(x)
    grad_in.index_add_(0, idx, grad_out)
    return grad_in

grad_in = getitem_back_rows(grad_out, x, idx)
print(grad_in.tolist())
print('row 1 summed two grads:', grad_in[1].tolist() == [5.0, 5.0, 5.0])